In [1]:
# ======================================================
# CIND820 - Keystone Project
# Data Analysis and Visualization: Fire Peril Loss Cost (Liberty Mutual Insurance)
# Author: Debra Allen 
# ======================================================

# Call Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [2]:
# =======================================================
# Data Preparation for Modeling: Fire Peril Loss Cost
# =======================================================

# Import necessary libraries for data preparation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# ======================================================
# Load the dataset and display basic information
# ======================================================

# Use a smaller sample for modeling to avoid memory issues
def find_repo_root(start: Path | None = None) -> Path:
    """Find the root of the git repository."""
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    
    markers = [".git", "pyproject.toml", "requirements.txt"]
    for p in [start, *start.parents]:
        if any((p / m).exists() for m in markers):
            return p
    raise FileNotFoundError("Could not find repo root. Run notebook from inside the repo.")

REPO_ROOT = find_repo_root()
SAMPLE_PATH = REPO_ROOT /"liberty_train_subset.csv"
print("Repo Root:", REPO_ROOT)
print("Data Path:", SAMPLE_PATH)

# Load a smaller sample of the dataset for modeling
model_data = pd.read_csv(SAMPLE_PATH, low_memory=False)

# Display basic information about the dataset
model_data.info()

Repo Root: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820
Data Path: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820\liberty_train_subset.csv
<class 'pandas.DataFrame'>
RangeIndex: 113015 entries, 0 to 113014
Columns: 302 entries, id to weatherVar236
dtypes: float64(291), int64(1), str(10)
memory usage: 260.4 MB


In [4]:
# ======================================================
# Prepare the dataset for modeling
# ======================================================

# Convert categorical variables to string type (if any)
for col in model_data.select_dtypes(include=["object"]).columns:
    model_data[col] = model_data[col].astype(str)

# Define the target variable and predictor variables
target = "target"
X = model_data.drop(columns=[target])
y = model_data[target]

C:\Users\uni_f\AppData\Local\Temp\ipykernel_18524\2822705644.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in model_data.select_dtypes(include=["object"]).columns:


In [5]:
# ======================================================
# Identify Variable Types
# ======================================================

# Identify categorical and numerical variables
categorical_vars = X.select_dtypes(include=["object"]).columns.tolist()
numerical_vars = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Display variable types
print("Categorical Variables:", categorical_vars)
print("Numerical Variables:", numerical_vars)

# Display number of numerical and categorical variables
print("\nNumber of Categorical Variables:", len(categorical_vars))
print("Number of Numerical Variables:", len(numerical_vars))

Categorical Variables: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'dummy']
Numerical Variables: ['id', 'var10', 'var11', 'var12', 'var13', 'var14', 'var15', 'var16', 'var17', 'crimeVar1', 'crimeVar2', 'crimeVar3', 'crimeVar4', 'crimeVar5', 'crimeVar6', 'crimeVar7', 'crimeVar8', 'crimeVar9', 'geodemVar1', 'geodemVar2', 'geodemVar3', 'geodemVar4', 'geodemVar5', 'geodemVar6', 'geodemVar7', 'geodemVar8', 'geodemVar9', 'geodemVar10', 'geodemVar11', 'geodemVar12', 'geodemVar13', 'geodemVar14', 'geodemVar15', 'geodemVar16', 'geodemVar17', 'geodemVar18', 'geodemVar19', 'geodemVar20', 'geodemVar21', 'geodemVar22', 'geodemVar23', 'geodemVar24', 'geodemVar25', 'geodemVar26', 'geodemVar27', 'geodemVar28', 'geodemVar29', 'geodemVar30', 'geodemVar31', 'geodemVar32', 'geodemVar33', 'geodemVar34', 'geodemVar35', 'geodemVar36', 'geodemVar37', 'weatherVar1', 'weatherVar2', 'weatherVar3', 'weatherVar4', 'weatherVar5', 'weatherVar6', 'weatherVar7', 'weatherVar8', 'weatherVar9

C:\Users\uni_f\AppData\Local\Temp\ipykernel_18524\2312351895.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_vars = X.select_dtypes(include=["object"]).columns.tolist()


In [6]:
# ======================================================
# Preprocess the dataset
# ======================================================

# Numerical pipeline: impute missing values with median
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical pipeline: impute missing values with most frequent and one-hot encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine numerical and categorical pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_vars),
    ('cat', categorical_pipeline, categorical_vars)
],
remainder='drop',
n_jobs=-1
)

In [7]:
# =======================================================
# Train-Test Split
# =======================================================

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])


Training observations: 90412
Testing observations: 22603


In [8]:
# ======================================================
# Gradient Boosting Model Specification
# ======================================================

gradient_boosting_model = GradientBoostingRegressor(
    n_estimators=200,  # Number of boosting stages
    learning_rate=0.05,  # Learning rate
    max_depth=3,  # Maximum depth of the individual regression estimators
    subsample=0.8,  # Subsample ratio of the training instances
    random_state=42
)

In [9]:
# ======================================================
# Full Modeling Pipeline
# ======================================================

gbr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', gradient_boosting_model)
])

In [10]:
# ======================================================
# Model Training
# ======================================================

gbr_pipeline.fit(X_train, y_train)

print("Gradient Boosting Model fitted successfully.")

Gradient Boosting Model fitted successfully.


In [11]:
# ======================================================
# Model Evaluation
# ======================================================

# Predict on the test set
y_pred = gbr_pipeline.predict(X_test)

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)  # RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display evaluation results
print("\nGradient Boosting Performance (Test Dataset):")
print("-------------------------------------------------")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R2): {r2:.4f}")


Gradient Boosting Performance (Test Dataset):
-------------------------------------------------
Mean Squared Error (MSE): 0.0644
Mean Absolute Error (MAE): 0.0155
R-squared (R2): -0.0289


In [13]:
# ======================================================
# Gradient Boosting Model Interpretation
# ======================================================

# Extract model coefficients after preprocessing
var_names_num = numerical_vars
var_names_cat = gbr_pipeline.named_steps['preprocessor'] \
    .named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_vars)

feature_names = np.concatenate([var_names_num, var_names_cat])

# Extract feature importances from the model
importances = gbr_pipeline.named_steps['model'].feature_importances_

# Create a DataFrame for coefficients
importance_table = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

importance_table.head(20)  # Display top 20 features by importance value

,Feature,Importance
349,var4_S1,0.096249
4,var13,0.091328
348,var4_R8,0.055911
6,var15,0.047300
2,var11,0.038204
0,id,0.037538
336,var4_M1,0.027664
109,weatherVar55,0.025969
367,var7_6,0.021304
366,var7_5,0.020447


**Gradient Boosting Summary**

A gradient boosting regression model was fitted to capture non-linear effects and interactions among predictors. Compared to the Tweedie GLM, the gradient boosting model offers greater flexibility with reduced interpretability. In the baseline configuration, performance gains were limited due to extreme zero-inflation and weak signal strength which highlights the inherent difficulty of fire-peril loss cost prediction.